# core

> Typed questions, gold targets, the Jev answer schema, rows and batches: the small core every other layer builds on.

In [ ]:
#| default_exp core

In [ ]:
#| export
import json, math
from dataclasses import dataclass, field
from functools import singledispatch
from pathlib import Path

import numpy as np
import torch
from fastcore.basics import patch

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail
from nbdev.showdoc import show_doc

sysone answers *typed questions* about a *state*. A state is whatever the model reads: a string, a JSON object, or, for the multimodal application, text plus a screenshot. A question has a type, an instruction and, depending on the type, criteria. The canonical record is a case from [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions), which is also the body of a Jev `POST /v1/systemone` request plus its gold answers. The fixtures folder holds twenty of them:

In [ ]:
recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
rec = recs[0]
print(json.dumps(rec["state"], indent=1)[:400])
list(rec["questions"]), rec["questions"]["action"]

## The path of a decision

Every layer of the library sits somewhere on one path, from a state and its questions to answers in the Jev schema:

```{mermaid}
flowchart LR
    S["state<br/>text, JSON, a screenshot"] --> RB
    Q["questions<br/>choice · score · noul"] --> RB["RowBuilder<br/>one row per question,<br/>an anchor before every option"]
    RB --> C["collate<br/>one batch"]
    C --> E["encoder<br/>one forward pass"]
    E --> H["DecisionHead<br/>one logit per anchor"]
    H --> T["÷ temperature,<br/>softmax per row"]
    T --> A["fill_answer<br/>the Jev schema"]
```

A question never generates text: its options are fixed before the model runs, each one behind an anchor token, and the answer is read off the probabilities at those anchors by code. This notebook holds the two ends of the path, the questions and answers, and the containers in between, rows and batches; the templates, the model, the losses and the loader each have a notebook of their own.

## Question types

The three primitives, and the integer each one is embedded as. The order is Laya's, so a head trained here reads the same type embedding row as a Laya checkpoint, and Laya-format exports stay loadable.

In [ ]:
#| export
QTYPES = {"choice": 0, "score": 1, "noul": 2}
QTYPE_NAMES = {v: k for k, v in QTYPES.items()}
NOUL_KEYS = ("false", "true")

In [ ]:
#| export
def _check_label(qname, i, label):
    if isinstance(label, (list, dict, set, bytearray)):
        raise ValueError(f"question {qname!r}: choice label {i} is a {type(label).__name__}; a label is rendered "
                         f"as option text and used as the answer key, so it must be a scalar, got {label!r}")

def _check_question(qname, type, instructions, criteria, labels):
    "Reject a question that cannot be answered, naming it and what to fix (Laya's rules)"
    if type not in QTYPES: raise ValueError(f"question {qname!r}: unknown type {type!r}; use one of {sorted(QTYPES)}")
    if instructions is None: raise ValueError(f"question {qname!r}: no 'instructions'; add the text the model should answer")
    if type == "choice":
        if not isinstance(criteria, (dict, list)) or not criteria:
            raise ValueError(f"question {qname!r}: a choice question takes 'criteria' as a non-empty dict of "
                             "label -> description, or a list of labels")
        for i, label in enumerate(criteria if isinstance(criteria, list) else criteria.keys()): _check_label(qname, i, label)
    elif type == "score":
        if not isinstance(criteria, list) or not criteria:
            raise ValueError(f"question {qname!r}: a score question takes 'criteria' as a non-empty list of level "
                             "descriptions, index 0 first")
        if None in criteria: raise ValueError(f"question {qname!r}: score level {criteria.index(None)} is null")
    elif criteria is not None:
        if not isinstance(criteria, dict) or not {str(k).lower() for k in criteria} <= set(NOUL_KEYS):
            raise ValueError(f"question {qname!r}: a noul question takes 'criteria' keyed only 'true'/'false' "
                             "(either or both), or omits it")
    if labels is not None:
        if type != "noul": raise ValueError(f"question {qname!r}: 'labels' is only supported for noul questions")
        if (not isinstance(labels, dict) or set(labels) != set(NOUL_KEYS)
            or not all(isinstance(v, str) and v.strip() for v in labels.values())
            or labels["false"].strip() == labels["true"].strip()):
            raise ValueError(f"question {qname!r}: noul labels must map exactly 'false' and 'true' to distinct non-empty strings")

In [ ]:
#| export
class Question:
    "One typed question over a state: a `choice` among labels, an ordered `score` rubric, or a yes/no `noul`"
    def __init__(self,
                 type:str, # "choice", "score" or "noul"
                 instructions:str, # the text the model answers
                 criteria=None, # choice: {label: description} or [labels]; score: [level descriptions]; noul: {"false": .., "true": ..} or None
                 labels:dict|None=None, # noul only: the words shown for false and true, e.g. {"false": "no", "true": "yes"}
                 name:str|None=None): # the question's id within its case
        _check_question(name, type, instructions, criteria, labels)
        if type == "choice" and isinstance(criteria, list): criteria = {c: None for c in criteria}
        if type == "noul" and criteria is not None: criteria = {str(k).lower(): v for k, v in criteria.items()}
        if not isinstance(instructions, str): instructions = json.dumps(instructions, ensure_ascii=False)
        self.type, self.instructions, self.criteria, self.labels, self.name = type, instructions, criteria, labels, name

    @classmethod
    def from_dict(cls, d:dict, name=None):
        "A `Question` from its Jev/typed-decisions definition"
        if isinstance(d, Question): return d
        if not isinstance(d, dict): raise ValueError(f"question {name!r}: definition must be a dict, got {type(d).__name__}")
        return cls(d.get("type"), d.get("instructions"), d.get("criteria"), d.get("labels"), name=name)

    def __repr__(self):
        ins = self.instructions if len(self.instructions) < 40 else self.instructions[:39] + "…"
        return f"Question({self.name or '?'}: {self.type}, {ins!r}, {self.k} options)"

A question is built from its definition exactly as it appears in the record. Choice criteria given as a list become labels without descriptions, noul criteria keys are normalised to `"false"`/`"true"`:

In [ ]:
q = Question.from_dict(rec["questions"]["action"], name="action")
q

In [ ]:
test_eq(Question("choice", "Pick one", ["a", "b"]).criteria, {"a": None, "b": None})
test_eq(Question("noul", "Is it?", {True: "yes it is"}).criteria, {"true": "yes it is"})
test_fail(lambda: Question("choice", "Pick one", {}), contains="non-empty")
test_fail(lambda: Question("score", "Rate", {"a": 1}), contains="list of level")
test_fail(lambda: Question("noul", "Is it?", {"maybe": "x"}), contains="keyed only")
test_fail(lambda: Question("choice", "x", ["a"], labels={"false": "n", "true": "y"}), contains="only supported for noul")
test_fail(lambda: Question("bool", "x"), contains="unknown type")

### Options and answer keys

Every question becomes an ordered list of *options*, the texts that sit behind the anchor tokens in a row, and the same number of *keys*, the names the answer is reported under. A choice's keys are its labels, a score's are the level indices as strings, a noul's are always `false` then `true`. The option texts follow Laya's rendering: `label: description` for choices, `level i: description` for scores, and `false: …`, `true: …` for nouls, with Laya's default descriptions when none is given.

In [ ]:
#| export
def render_criterion(value) -> str:
    "A criterion value as text: strings pass through, anything structured becomes compact JSON"
    if isinstance(value, str): return value
    return json.dumps(value, ensure_ascii=False, separators=(", ", ": "), default=str)

_NOUL_DEFAULTS = {"false": "no, the statement does not hold", "true": "yes, the statement holds"}

@patch(as_prop=True)
def qtype(self:Question) -> int: "The type's integer id"; return QTYPES[self.type]

@patch(as_prop=True)
def keys(self:Question) -> list:
    "Answer keys, in option order"
    if self.type == "choice": return list(self.criteria)
    if self.type == "score": return [str(i) for i in range(len(self.criteria))]
    return list(NOUL_KEYS)

@patch(as_prop=True)
def k(self:Question) -> int: "Number of options"; return len(self.keys)

@patch(as_prop=True)
def options(self:Question) -> list:
    "Option texts in key order, rendered as Laya renders them"
    if self.type == "choice":
        return [str(k) if v is None or v == "" else f"{k}: {render_criterion(v)}" for k, v in self.criteria.items()]
    if self.type == "score": return [f"level {i}: {render_criterion(c)}" for i, c in enumerate(self.criteria)]
    crit, labels = self.criteria or {}, self.labels or {"false": "false", "true": "true"}
    return [f"{labels[key].strip()}: " + (render_criterion(crit[key]) if crit.get(key) not in (None, "") else _NOUL_DEFAULTS[key])
            for key in NOUL_KEYS]

In [ ]:
q.keys, q.options

In [ ]:
urg = Question.from_dict(rec["questions"]["urgency"], name="urgency")
test_eq(urg.keys, ["0", "1", "2", "3"])
test_eq(urg.options[0], "level 0: No time pressure; can wait indefinitely.")
nq = Question("noul", "The invoice is a duplicate.")
test_eq(nq.options, ["false: no, the statement does not hold", "true: yes, the statement holds"])
test_eq(Question("noul", "Is it?", labels={"false": "B", "true": "A"}).options[1], "A: yes, the statement holds")
test_eq(Question("choice", "Pick", [1, 2]).options, ["1", "2"])
test_eq(Question("choice", "Pick", {"a": {"desc": "x"}}).options, ['a: {"desc": "x"}'])

In [ ]:
#| export
@patch
def to_dict(self:Question) -> dict:
    "The Jev definition of this question"
    d = {"type": self.type, "instructions": self.instructions}
    if self.criteria is not None: d["criteria"] = self.criteria if self.type != "score" else list(self.criteria)
    if self.labels is not None: d["labels"] = self.labels
    return d

def as_questions(questions) -> dict:
    "`{qid: Question}` from a mapping of definitions (or `Question`s)"
    if isinstance(questions, str): questions = json.loads(questions)
    return {qid: Question.from_dict(q, name=qid) for qid, q in questions.items()}

In [ ]:
qs = as_questions(rec["questions"])
test_eq(list(qs), list(rec["questions"]))
test_eq(Question.from_dict(qs["risk"].to_dict()).options, qs["risk"].options)
qs

## Gold targets

Training needs, for every question, a distribution over its options. Gold may be a label, a distribution, or both: typed-decisions ships the mean of three teacher samples as `probabilities` plus the argmax `label`, and converters from classification datasets only have labels. `Question.target` turns any of these into the target distribution (in option order) and the index of the gold label, the notebook's `build_training_item`. When a label is present its index is kept for accuracy, otherwise the argmax of the distribution is used.

In [ ]:
#| export
def _two_point(score, n):
    "A distribution over n levels whose expected level is `score`"
    s = min(max(float(score), 0.0), n - 1.0)
    lo = int(math.floor(s)); p = [0.0] * n
    if lo >= n - 1: p[n - 1] = 1.0
    else: p[lo], p[lo + 1] = 1.0 - (s - lo), s - lo
    return p

def _noul_key(v) -> str:
    "A truth value's key: 'true' or 'false' for yes/no, booleans, 1/0"
    s = str(v).strip().lower()
    return {"yes": "true", "no": "false", "1": "true", "0": "false", "1.0": "true", "0.0": "false"}.get(s, s)

def _str_key(k) -> str:
    "A key as text, integral floats without their '.0'"
    return str(int(k)) if isinstance(k, float) and k.is_integer() else str(k)

def _key_index(q, label):
    "Index of a gold label among the question's keys, or -1"
    if label is None: return -1
    if q.type == "noul":
        s = _noul_key(label)
        return NOUL_KEYS.index(s) if s in NOUL_KEYS else -1
    keys = [_str_key(k) for k in q.keys]
    return keys.index(_str_key(label)) if _str_key(label) in keys else -1

@patch
def target(self:Question, gold) -> tuple:
    "The gold distribution over options (key order) and the gold label's index; `(None, -1)` without gold"
    if gold is None: return None, -1
    if isinstance(gold, (list, tuple)) and len(gold) == self.k and all(isinstance(v, (int, float)) for v in gold):
        t, label = [float(v) for v in gold], -1
    else:
        if not isinstance(gold, dict):   # a bare value: a label, or an expected score or p(true) when fractional
            frac = isinstance(gold, float) and not gold.is_integer()
            gold = {"score": gold} if frac and self.type == "score" else {"noul": gold} if frac and self.type == "noul" else {"label": gold}
        label, probs, t = _key_index(self, gold.get("label")), gold.get("probabilities"), None
        if isinstance(probs, dict) and probs:
            if self.type == "noul":
                pn = {_noul_key(k): v for k, v in probs.items()}
                pt, pf = pn.get("true"), pn.get("false")
                if pt is not None or pf is not None:
                    pt = 1.0 - pf if pt is None else pt
                    pf = 1.0 - pt if pf is None else pf
                    t = [float(pf), float(pt)]
            else:
                pk = {_str_key(k): v for k, v in probs.items()}
                t = [float(pk.get(_str_key(k), 0.0)) for k in self.keys]
            if t is not None and sum(t) <= 0: t = None     # the distribution names none of the options
        if t is not None: pass
        elif self.type == "noul" and gold.get("noul") is not None: t = [1.0 - float(gold["noul"]), float(gold["noul"])]
        elif self.type == "score" and gold.get("score") is not None: t = _two_point(gold["score"], self.k)
        elif label >= 0: t = [float(i == label) for i in range(self.k)]
        else: return None, -1
    s = sum(t)
    t = [v / s for v in t] if s > 0 else [1.0 / len(t)] * len(t)
    return t, (label if label >= 0 else int(np.argmax(t)))

In [ ]:
t, label = q.target(rec["gold"]["action"])
test_close(sum(t), 1.0)
test_eq(q.keys[label], rec["gold"]["action"]["label"])
t, label

A record can also say that none of the offered options answers the question: gold `{"answerable": false}`. It has no target, so the option loss leaves it out, but it is not a missing label either. `answerable` tells the three cases apart, and a head with an act output learns from the unanswerable ones to hand the case over rather than answer (see [models](03_models.ipynb#acting-or-escalating)):

In [ ]:
#| export
@patch
def answerable(self:Question, gold) -> int:
    "1 if the gold answers the question with its options, 0 if it says none of them does (`{'answerable': False}`), -1 without gold"
    if gold is None: return -1
    if isinstance(gold, dict) and gold.get("answerable") is False: return 0
    return 1 if self.target(gold)[0] is not None else -1

In [ ]:
test_eq(q.answerable({"answerable": False}), 0)
test_eq(q.target({"answerable": False}), (None, -1))
test_eq((q.answerable("stop"), q.answerable(None), q.answerable({"label": "not an option"})), (1, -1, -1))

The other gold shapes: a bare label, a noul probability, an expected score (spread over the two nearest levels so the expectation is preserved), a distribution in option order.

In [ ]:
test_eq(q.target("stop"), ([0.0, 0.0, 0.0, 1.0], 3))
test_eq(nq.target({"noul": 0.8}), ([0.19999999999999996, 0.8], 1))
test_eq(nq.target(True)[1], 1)
test_close(urg.target({"score": 1.4})[0], [0, 0.6, 0.4, 0])
test_eq(urg.target([0, 1, 1, 0]), ([0, 0.5, 0.5, 0], 1))
test_eq(q.target(None), (None, -1))

Truth values may be written yes/no, booleans or 1/0, integer keys and integral floats name the same option as their strings, a fractional bare value is an expected score or a p(true), and a distribution that names none of the options is no gold at all rather than a uniform target:

In [ ]:
test_eq(nq.target({"probabilities": {"yes": 0.8, "no": 0.2}}), ([0.2, 0.8], 1))
test_eq(nq.target({"probabilities": {True: 0.7}})[0], [0.30000000000000004, 0.7])
test_eq(urg.target({"probabilities": {0: 0.1, 1: 0.9}}), ([0.1, 0.9, 0.0, 0.0], 1))
test_eq(urg.target(2.0), ([0.0, 0.0, 1.0, 0.0], 2))
test_close(urg.target(1.4)[0], [0, 0.6, 0.4, 0])
test_eq(nq.target(0.9)[1], 1)
test_eq(q.target({"probabilities": {"elsewhere": 1.0}}), (None, -1))
test_eq(q.target({"probabilities": {"elsewhere": 1.0}, "label": "stop"}), ([0.0, 0.0, 0.0, 1.0], 3))

## The answer schema

The model returns one probability per option; code reads the answer off it, as Jev and Laya do, so nothing is generated and nothing is parsed. Two confidences are reported. `confidence` is Laya's normalised-entropy confidence, $1 - H(p)/\log k$, how concentrated the whole distribution is. `answer_confidence` is $\max p$, the probability of the reported answer, which is the quantity temperature scaling fits and ECE measures, so it is the one to gate on.

In [ ]:
#| export
def entropy_confidence(p) -> float:
    "1 − H(p)/log k: how concentrated the distribution is"
    p = np.asarray(p, dtype=np.float64); k = len(p)
    if k < 2: return 1.0
    ent = -(p * np.log(np.clip(p, 1e-12, 1.0))).sum()
    return float(np.clip(1.0 - ent / math.log(k), 0.0, 1.0))

def answer_confidence(p) -> float:
    "max(p): the probability of the reported answer"
    p = np.asarray(p, dtype=np.float64)
    return float(np.clip(p.max(), 0.0, 1.0)) if len(p) else 1.0

In [ ]:
test_eq(entropy_confidence([0.25] * 4), 0.0)
test_eq(entropy_confidence([1.0, 0.0, 0.0]), 1.0)
test_close(entropy_confidence([0.7, 0.2, 0.1]), 0.2702, eps=1e-4)
test_eq(answer_confidence([0.7, 0.2, 0.1]), 0.7)

The two can disagree on which of two answers is the surer. A tie between two options is concentrated, half the options having no mass at all, yet its answer is a coin toss; a weak favourite among four is spread out, yet its answer is the likelier to be right:

In [ ]:
tie, favourite = [0.5, 0.5, 0.0, 0.0], [0.55, 0.15, 0.15, 0.15]
test_eq(entropy_confidence(tie) > entropy_confidence(favourite), True)
test_eq(answer_confidence(tie) < answer_confidence(favourite), True)
{"tie": (round(entropy_confidence(tie), 3), answer_confidence(tie)), "favourite": (round(entropy_confidence(favourite), 3), answer_confidence(favourite))}

So `answer_confidence` is the number to gate an answer on, and `confidence` is the better sign of ambiguity among a few options.

In [ ]:
#| export
class Answer(dict):
    "A typed answer in the Jev schema: a `dict`, with attribute access for convenience"
    def __getattr__(self, k):
        try: return self[k]
        except KeyError: raise AttributeError(k) from None

def _r(x, digits): return round(float(x), digits) if digits is not None else float(x)

def fill_answer(q:Question, probs, digits:int|None=4, act:float|None=None, act_threshold:float|dict|None=None) -> Answer:
    "The Jev answer for `q` from its option probabilities: the argmax choice, the expected score, or p(true); `act`, from an act head, adds `action`"
    p = np.asarray(probs, dtype=np.float64)[:q.k]
    ent, top = _r(entropy_confidence(p), digits), _r(answer_confidence(p), digits)
    if q.type == "choice":
        a = Answer(type="choice", choice=q.keys[int(p.argmax())],
                   probabilities={k: _r(v, digits) for k, v in zip(q.keys, p)},
                   confidence=ent, answer_confidence=top)
    elif q.type == "score":
        a = Answer(type="score", score=_r((np.arange(q.k) * p).sum(), digits),
                   legend={str(i): c for i, c in enumerate(q.criteria)},
                   probabilities={str(i): _r(v, digits) for i, v in enumerate(p)},
                   confidence=ent, answer_confidence=top)
    else: a = Answer(type="noul", noul=_r(p[1], digits), confidence=_r(max(p[1], 1 - p[1]), digits), answer_confidence=top)
    if act is not None:   # Laya's `action` field: whether to act on the answer or hand the case to a person
        thr = act_threshold if act_threshold is None or isinstance(act_threshold, (int, float)) else lookup_threshold(act_threshold, q.qtype, q.k)
        a["action"] = {"act_probability": _r(act, digits)} | ({"act": bool(act >= thr)} if thr is not None else {})
    return a

In [ ]:
fill_answer(q, [0.1, 0.6, 0.2, 0.1])

In [ ]:
a = fill_answer(urg, [0.1, 0.2, 0.3, 0.4])
test_eq(a.score, 2.0)
test_eq(a.legend["3"], urg.criteria[3])
b = fill_answer(nq, [0.25, 0.75])
test_eq((b.noul, b.confidence, b.answer_confidence), (0.75, 0.75, 0.75))
test_eq(fill_answer(q, [0.1, 0.6, 0.2, 0.1]).choice, "human_review")
test_eq(json.loads(json.dumps(b)), dict(b))

A model with an act head adds Laya's `action` field: the probability that it should act on this answer rather than escalate the case, and, once a threshold has been fitted on held-out rows, the decision itself. The threshold is one number, or one per question type and option-count bucket (see [below](#temperature-buckets)):

In [ ]:
a = fill_answer(q, [0.1, 0.6, 0.2, 0.1], act=0.83, act_threshold=0.7)
test_eq(a.action, {"act_probability": 0.83, "act": True})
test_eq("action" in fill_answer(q, [0.1, 0.6, 0.2, 0.1]), False)
a.action

## Temperature buckets

Calibration fits one temperature per question type and option-count bucket (2, 3 to 5, 6 to 10, 11+), named as Laya names them, so the same keys work in `sysone.json` and in `rl_agent_config.json`. A missing bucket falls back to the type's temperature, then to 1.

In [ ]:
#| export
def temp_bucket(qtype, k:int) -> str:
    "The calibration bucket of a question type (name or id) with k options, e.g. `choice:3-5`"
    name = qtype if isinstance(qtype, str) else QTYPE_NAMES[int(qtype)]
    size = "2" if k <= 2 else "3-5" if k <= 5 else "6-10" if k <= 10 else "11+"
    return f"{name}:{size}"

TEMP_MIN, TEMP_MAX = 0.5, 5.0

def clamp_temperature(t, lo=TEMP_MIN, hi=TEMP_MAX) -> float:
    "A usable temperature: `t` confined to [lo, hi], or 1.0 if it is not a finite number"
    try: t = float(t)
    except (TypeError, ValueError): return 1.0
    if not math.isfinite(t): return 1.0
    return min(hi, max(lo, t))

def lookup_temperature(temps:dict|None, qtype, k:int) -> float:
    "The temperature for a row: its bucket, else its type, else 1"
    if not temps: return 1.0
    name = qtype if isinstance(qtype, str) else QTYPE_NAMES[int(qtype)]
    return clamp_temperature(temps.get(temp_bucket(name, k), temps.get(name, 1.0)))

In [ ]:
test_eq([temp_bucket("choice", k) for k in (2, 4, 7, 20)], ["choice:2", "choice:3-5", "choice:6-10", "choice:11+"])
test_eq(temp_bucket(2, 2), "noul:2")
temps = {"choice:3-5": 1.4, "choice": 1.1, "score:3-5": 0.1}
test_eq(lookup_temperature(temps, "choice", 4), 1.4)
test_eq(lookup_temperature(temps, "choice", 7), 1.1)
test_eq(lookup_temperature(temps, "score", 4), 0.5)
test_eq(lookup_temperature(None, "noul", 2), 1.0)

Act thresholds are keyed the same way. A bucket's threshold applies first, then its type's, then the one fitted over all rows (`all`), so a question kind whose act scores run lower than the others' still gets a threshold of its own:

In [ ]:
#| export
def lookup_threshold(thresholds, qtype, k:int) -> float|None:
    "The act threshold for a row: a single number, or from a dict its bucket's, else its type's, else the one over all rows (`all`)"
    if thresholds is None or isinstance(thresholds, (int, float)): return None if thresholds is None else float(thresholds)
    name = qtype if isinstance(qtype, str) else QTYPE_NAMES[int(qtype)]
    t = thresholds.get(temp_bucket(name, k), thresholds.get(name, thresholds.get("all")))
    return None if t is None else float(t)

In [ ]:
thr = {"all": 0.8, "choice": 0.7, "choice:3-5": 0.6}
test_eq([lookup_threshold(thr, "choice", 4), lookup_threshold(thr, "choice", 8), lookup_threshold(thr, "noul", 2)], [0.6, 0.7, 0.8])
test_eq((lookup_threshold(0.5, "noul", 2), lookup_threshold(None, "noul", 2), lookup_threshold({"choice": 0.7}, "noul", 2)), (0.5, None, None))
test_eq(fill_answer(q, [0.1, 0.6, 0.2, 0.1], act=0.65, act_threshold=thr).action, {"act_probability": 0.65, "act": True})    # its bucket's 0.6
test_eq(fill_answer(nq, [0.25, 0.75], act=0.65, act_threshold=thr).action["act"], False)                                    # 'all': 0.8

## States

A state is rendered to the text the encoder reads: strings pass through and anything else becomes JSON, exactly as Laya serialises it, so rows built here match rows Laya builds. `render_state` is a `singledispatch` function; register a type to render it differently (the transforms in `sysone.template` are the usual way to shape a state, this is the fallback).

In [ ]:
#| export
@singledispatch
def render_state(state) -> str:
    "The text of a state: JSON for structured states"
    return json.dumps(state, ensure_ascii=False)

@render_state.register
def _(state:str) -> str: return state

In [ ]:
test_eq(render_state("plain text"), "plain text")
test_eq(render_state({"a": "é", "b": [1, 2]}), '{"a": "é", "b": [1, 2]}')

## Records

A *record* is one case: a state, its questions, and (for training) the gold answers. Hub datasets often store the three as JSON strings; `load_record` parses them, leaves everything else in place, and checks the questions. A state passed at inference is taken as it is (`parse_state=False`), as Laya's runtime takes it: a string that happens to be JSON stays a string.

In [ ]:
#| export
def _parse_state(v):
    "A stored state: JSON when it parses, the text itself otherwise"
    if isinstance(v, str):
        try: return json.loads(v)
        except json.JSONDecodeError: return v
    return v

def load_record(r:dict, parse_state:bool=True) -> dict:
    "A case record with `questions` and `gold` parsed from JSON strings, and the `state` too when `parse_state`"
    r = dict(r)
    if "state" not in r or "questions" not in r: raise ValueError("a record needs 'state' and 'questions'")
    if parse_state: r["state"] = _parse_state(r["state"])
    r["questions"] = json.loads(r["questions"]) if isinstance(r["questions"], str) else r["questions"]
    if isinstance(r.get("gold"), str): r["gold"] = json.loads(r["gold"])
    as_questions(r["questions"])
    return r

In [ ]:
hub_row = {"id": "x", "state": json.dumps(rec["state"]), "questions": json.dumps(rec["questions"]), "gold": json.dumps(rec["gold"])}
test_eq(load_record(hub_row)["state"], rec["state"])
test_eq(load_record({"state": "text", "questions": {}})["state"], "text")
test_eq(load_record({"state": '{"a":1}', "questions": {}}, parse_state=False)["state"], '{"a":1}')
test_fail(lambda: load_record({"state": "x", "questions": {"q": {"type": "choice", "instructions": "x"}}}), contains="choice")

## Rows

One question over one state is one *row*: token ids, the positions of the anchor before each option (`markers`), the question type, and, for training, the target distribution and label. Everything a multimodal encoder needs beyond ids (`pixel_values`, `position_ids`, `image_grid_hw`, or a side stream's token ids) rides in `extras`, and so do cached encoder outputs (`anchors`, the vectors at the markers, or `hidden`, the whole sequence), so nothing downstream of the encoder needs to know the modality. A row also records where each option's own tokens are (`spans`, for heads that read an option as the mean of its tokens rather than at its anchor), and whether its gold answers the question at all (`answerable`, for the act head). `sysone.template.RowBuilder` makes rows; this is only the container.

In [ ]:
#| export
ROW_META = ("n_tokens", "case", "qid", "case_idx", "variant")   # bookkeeping columns of a row: never model inputs

@dataclass
class Row:
    "One question over one state, as token ids with the anchor position of every option"
    ids: list # token ids
    markers: list # position of the anchor before each option
    qtype: int # QTYPES id
    target: list|None = None # gold distribution over the options
    label: int = -1 # index of the gold option
    extras: dict = field(default_factory=dict) # pixel_values, position_ids, image_grid_hw, stream ids, features
    meta: dict = field(default_factory=dict) # case id, question id
    spans: list|None = None # (start, end) of each option's own tokens, after its anchor
    answerable: int = -1 # 1: the gold is among the options; 0: the record says none is; -1: no gold

    @property
    def n_tokens(self): return len(self.ids)
    @property
    def k(self): return len(self.markers)

    def to_dict(self) -> dict:
        "The row as dataset columns"
        d = {"input_ids": list(self.ids), "markers": list(self.markers), "qtype": self.qtype,
             "target": self.target, "label": self.label, "n_tokens": self.n_tokens, "answerable": self.answerable, **self.extras}
        if self.spans is not None: d["spans"] = [list(s) for s in self.spans]
        return d

    @classmethod
    def from_dict(cls, d:dict):
        "A row from dataset columns"
        known = {"input_ids", "markers", "qtype", "target", "label", "spans", "answerable", *ROW_META}
        spans = d.get("spans")
        return cls(list(d["input_ids"]), list(d["markers"]), int(d["qtype"]), d.get("target"), int(d.get("label", -1)),
                   extras={k: v for k, v in d.items() if k not in known},
                   meta={k: d[k] for k in ("case", "qid", "case_idx", "variant") if k in d},
                   spans=[tuple(int(x) for x in s) for s in spans] if spans is not None else None,
                   answerable=int(d.get("answerable", -1)))

In [ ]:
r = Row([1, 7, 2, 4, 9, 4, 8, 2, 5, 2], markers=[3, 5], qtype=2, target=[0.2, 0.8], label=1, spans=[(4, 5), (6, 8)], answerable=1)
test_eq((r.n_tokens, r.k), (10, 2))
test_eq(Row.from_dict(r.to_dict()), r)

## Batches

`collate` pads a list of rows into one batch: ids to the longest row, marker positions to the largest option count with `marker_mask` saying which are real, and targets zero-padded to match. Rows from different questions, cases and option counts mix freely; this is what lets several questions of one state run in a single forward pass. Extras are collated by kind: token-aligned tensors are padded along the sequence, per-image tensors are concatenated, cached features and side streams are padded to the batch. A kind can carry its pad value, `(kind, pad)`: a side stream's token ids pad with that tokenizer's own pad id, which need not be 0. Option spans are padded like the markers, and `answerable` rides along for the act head's loss and metrics.

In [ ]:
#| export
class Batch(dict):
    "A collated batch: `input_ids`, `attention_mask`, `marker_pos`, `marker_mask`, `qtype`, plus `target`/`label` and extras"
    def to(self, device):
        return Batch({k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in self.items()})

# how each extra is collated: "pad" along the last axis to the batch's length, "pad_rows" along the first axis to the
# batch's longest, "cat" along the first axis, "stack"; a kind may be a pair (kind, pad value)
EXTRA_COLLATE = {"position_ids": "pad", "token_type_ids": "pad", "pixel_values": "cat", "image_grid_hw": "cat",
                 "anchors": "pad_rows", "hidden": "pad_rows", "features": "pad_rows", "query": "stack", "first": "stack"}

def _as_tensor(v, dtype=None):
    return v.to(dtype) if isinstance(v, torch.Tensor) and dtype else v if isinstance(v, torch.Tensor) else torch.tensor(np.asarray(v), dtype=dtype)

def _collate_extra(name, vals, L):
    kind, pad = EXTRA_COLLATE.get(name, "stack"), 0
    if isinstance(kind, tuple): kind, pad = kind
    ts = [_as_tensor(v) for v in vals]
    if kind == "cat": return torch.cat(ts, 0)
    if kind == "pad":
        out = torch.full((len(ts), *ts[0].shape[:-1], L), pad, dtype=ts[0].dtype)
        for i, t in enumerate(ts): out[i, ..., :t.shape[-1]] = t
        return out
    if kind == "pad_rows":
        n = max(t.shape[0] for t in ts)
        out = torch.full((len(ts), n, *ts[0].shape[1:]), pad, dtype=ts[0].dtype)
        for i, t in enumerate(ts): out[i, :t.shape[0]] = t
        return out
    return torch.stack(ts)

def collate(rows, pad_id:int=0) -> Batch:
    "Pad rows (`Row`s or their dict form) into one `Batch`"
    rows = [r.to_dict() if isinstance(r, Row) else r for r in rows]
    n, L = len(rows), max(len(r["input_ids"]) for r in rows)
    K = max(len(r["markers"]) for r in rows)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, K), dtype=torch.long)
    mmask = torch.zeros((n, K), dtype=torch.bool)
    for i, r in enumerate(rows):
        ids[i, :len(r["input_ids"])] = _as_tensor(r["input_ids"], torch.long)
        att[i, :len(r["input_ids"])] = 1
        k = len(r["markers"])
        mpos[i, :k] = _as_tensor(r["markers"], torch.long)
        mmask[i, :k] = True
    b = Batch(input_ids=ids, attention_mask=att, marker_pos=mpos, marker_mask=mmask,
              qtype=torch.tensor([int(r["qtype"]) for r in rows]))
    if any(r.get("target") is not None for r in rows):
        tgt = torch.zeros((n, K), dtype=torch.float32)
        for i, r in enumerate(rows):
            if r.get("target") is None: continue
            if len(r["target"]) > len(r["markers"]):
                raise ValueError(f"row {i} has {len(r['target'])} target entries but {len(r['markers'])} markers")
            tgt[i, :len(r["target"])] = _as_tensor(r["target"], torch.float32)
        b["target"] = tgt
        b["label"] = torch.tensor([int(r.get("label", -1)) for r in rows])
    if any(r.get("spans") is not None for r in rows):
        sp = torch.zeros((n, K, 2), dtype=torch.long)
        for i, r in enumerate(rows):
            if r.get("spans"): sp[i, :len(r["spans"])] = _as_tensor(r["spans"], torch.long).reshape(-1, 2)
        b["spans"] = sp
    if any("answerable" in r for r in rows): b["answerable"] = torch.tensor([int(r.get("answerable", -1)) for r in rows])
    skip = {"input_ids", "markers", "qtype", "target", "label", "spans", "answerable", *ROW_META}
    for name in [k for k in rows[0] if k not in skip]:
        if all(r.get(name) is not None for r in rows): b[name] = _collate_extra(name, [r[name] for r in rows], L)
    return b

In [ ]:
r2 = Row([1, 5, 2, 4, 6, 4, 7, 4, 8, 2, 9, 9, 2], markers=[3, 5, 7], qtype=0, target=[0.1, 0.1, 0.8], label=2)
b = collate([r, r2], pad_id=3)
test_eq(b["input_ids"].shape, (2, 13))
test_eq(b["input_ids"][0, -1].item(), 3)
test_eq(b["marker_mask"].tolist(), [[True, True, False], [True, True, True]])
test_eq(b["target"][0].tolist(), [0.20000000298023224, 0.800000011920929, 0.0])
test_eq(b["label"].tolist(), [1, 2])
{k: tuple(v.shape) for k, v in b.items()}

Drawn cell by cell, a batch of three rows: the ids coloured by the part of the row they belong to, and the marker positions and mask that tell the head where the anchors are. The rows have different lengths and different option counts; the short rows are padded, and the padded option slots, hatched, point at position 0, with the mask keeping them out of the softmax. (The fixture's tokenizer is a 2,048-token BPE, so its words break into several pieces; a real tokenizer keeps most of them whole.)

![A batch of three rows: input_ids coloured by segment, and the marker positions and mask](images/batch.svg)

*Drawn from `RowBuilder` and `collate` in [figures](90_figures.ipynb#a-batch-of-rows).*

Extras are collated by kind; cached anchor vectors, for example, are padded to the batch's option count:

In [ ]:
fr = [Row([1, 2], [0], 0, extras={"anchors": np.ones((k, 4), dtype=np.float16)}) for k in (1, 3)]
test_eq(collate(fr)["anchors"].shape, (2, 3, 4))
pr = [Row([1, 2, 3][:n], [0], 0, extras={"position_ids": np.arange(n)[None].repeat(2, 0)}) for n in (2, 3)]
test_eq(collate(pr)["position_ids"].shape, (2, 2, 3))
EXTRA_COLLATE["side_input_ids"] = ("pad_rows", 1)       # a side stream whose tokenizer pads with id 1
sr = [Row([1, 2], [0], 0, extras={"side_input_ids": list(range(3, 3 + m))}, spans=[(1, 2)], answerable=a) for m, a in ((2, 1), (4, 0))]
sb = collate(sr)
test_eq(sb["side_input_ids"].tolist(), [[3, 4, 1, 1], [3, 4, 5, 6]])
test_eq((sb["spans"].shape, sb["answerable"].tolist()), ((2, 1, 2), [1, 0]))
del EXTRA_COLLATE["side_input_ids"]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()